In [ ]:
# 1. Mount Google Drive for your cookies.txt
from google.colab import drive
drive.mount('/content/drive')

# 2. Install libraries including curl_cffi for Cloudflare bypass
!pip install pyTelegramBotAPI curl_cffi -q
!pip install -U https://github.com/yt-dlp/yt-dlp/archive/master.zip -q

import telebot
from telebot.types import InlineKeyboardMarkup, InlineKeyboardButton
import yt_dlp
import os
import subprocess
import glob
import math

from google.colab import userdata
BOT_TOKEN = userdata.get('BOT_TOKEN')
ALLOWED_USERS = [int(userdata.get('USER_ID'))]
COOKIE_PATH = '/content/drive/MyDrive/BotData/cookies.txt'

bot = telebot.TeleBot(BOT_TOKEN)
user_urls = {}

@bot.message_handler(commands=['start'])
def send_welcome(message):
    if message.from_user.id in ALLOWED_USERS:
        bot.reply_to(message, "Bot API connected. Send a video link!")

@bot.message_handler(func=lambda message: True)
def handle_message(message):
    if message.from_user.id not in ALLOWED_USERS:
        return

    chat_id = message.chat.id
    if message.text.startswith("http"):
        user_urls[chat_id] = message.text

        markup = InlineKeyboardMarkup()
        btn_720 = InlineKeyboardButton("720p", callback_data="720")
        btn_480 = InlineKeyboardButton("480p", callback_data="480")
        btn_360 = InlineKeyboardButton("360p", callback_data="360")
        markup.add(btn_720, btn_480, btn_360)

        bot.send_message(chat_id, "Choose quality:", reply_markup=markup)

@bot.callback_query_handler(func=lambda call: True)
def handle_query(call):
    chat_id = call.message.chat.id
    if call.from_user.id not in ALLOWED_USERS:
        return

    bot.answer_callback_query(call.id, "Processing...")
    url = user_urls.get(chat_id)

    if not url:
        bot.edit_message_text("URL lost.", chat_id=chat_id, message_id=call.message.message_id)
        return

    target_height = call.data
    format_str = f'best[height<={target_height}]/best'

    ydl_opts = {
        'outtmpl': f'/content/media_{chat_id}.%(ext)s',
        'format': format_str,
        'quiet': True,
        'cookiefile': COOKIE_PATH,
        'impersonate': 'chrome', # Tricks the firewall into seeing a real Chrome browser
    }

    try:
        bot.edit_message_text("Downloading full video to Colab...", chat_id=chat_id, message_id=call.message.message_id)

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info_dict)
            video_duration = info_dict.get('duration', 0)

            # Extract the native title/tweet text. Fallback to "Video" if missing.
            video_title = info_dict.get('title', 'Video')

        file_size_mb = os.path.getsize(filename) / (1024 * 1024)

        # Telegram caps captions at 1024 characters. Keep title under 800 to leave room for the link and part numbers.
        safe_title = video_title[:800]

        if file_size_mb > 49:
            chunks_needed = math.ceil(file_size_mb / 45.0)
            segment_time = str(math.ceil(video_duration / chunks_needed)) if video_duration > 0 else "150"

            bot.edit_message_text(f"File is {file_size_mb:.1f}MB. Slicing into {chunks_needed} numbered parts...",
                                  chat_id=chat_id, message_id=call.message.message_id)

            split_pattern = f'/content/split_{chat_id}_%03d.mp4'
            subprocess.run(['ffmpeg', '-i', filename, '-c', 'copy', '-f', 'segment',
                            '-segment_time', segment_time, '-reset_timestamps', '1', split_pattern], check=True)

            split_files = sorted(glob.glob(f'/content/split_{chat_id}_*.mp4'))
            total_parts = len(split_files)

            bot.edit_message_text(f"Uploading {total_parts} parts to Telegram...", chat_id=chat_id, message_id=call.message.message_id)

            for index, part in enumerate(split_files, start=1):
                with open(part, 'rb') as f:
                    # Formats exactly as requested: Title Part X of Y
                    caption_text = f"{safe_title} part {index} of {total_parts}\nSource: {url}"
                    bot.send_video(chat_id, f, caption=caption_text)
                os.remove(part)

            os.remove(filename)
            bot.send_message(chat_id, "✅ All numbered parts uploaded!")

        else:
            bot.edit_message_text("Uploading to Telegram...", chat_id=chat_id, message_id=call.message.message_id)
            with open(filename, 'rb') as file_data:
                bot.send_video(chat_id, file_data, caption=f"{safe_title}\nSource: {url}")
            os.remove(filename)
            bot.send_message(chat_id, "✅ Upload complete!")

    except Exception as e:
        bot.send_message(chat_id, f"Failed: {str(e)}")

print("Smart Bot API with Titles is running...")
bot.infinity_polling()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Smart Bot API with Titles is running...
